# 01 — BQUANT Data Exploration & Factor Engine
## Personal Bloomberg BQUANT Replica

This notebook demonstrates the full research workflow:
1. Fetch market data for B3 (Brazilian) and US equities
2. Compute cross-sectional factor scores (momentum, low-vol)
3. Run a simple momentum backtest
4. Optimize a portfolio using PyPortfolioOpt

All data is fetch via `mybquant` — the platform's high-level API.

In [ ]:

# ── 1. Imports & Constants ─────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')   # allow importing mybquant from /notebooks

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Platform API
from mybquant import data, quant, portfolio

# Display settings
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_columns', 20)
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", palette="muted")

TRADING_DAYS = 252
RISK_FREE_RATE = 0.1175  # Brazilian Selic rate (annualized)

print("✓ Imports OK")
print(f"  Pandas {pd.__version__} | NumPy {np.__version__} | Seaborn {sns.__version__}")


## 2. Define Asset Universe

We focus on the **Ibovespa** universe (top 10 most liquid B3 stocks) and a few US reference ETFs.

In [ ]:

# ── 2. Universe Definition ────────────────────────────────────────────────
IBOVESPA = [
    "PETR4.SA",  # Petrobras
    "VALE3.SA",  # Vale
    "ITUB4.SA",  # Itaú Unibanco
    "BBDC4.SA",  # Bradesco
    "ABEV3.SA",  # Ambev
    "WEGE3.SA",  # WEG
    "RENT3.SA",  # Localiza
    "BBAS3.SA",  # Banco do Brasil
    "B3SA3.SA",  # B3
    "SUZB3.SA",  # Suzano
]

US_ETFS = ["SPY", "QQQ", "IWM", "EWZ"]  # EWZ = Brazil ETF (US-listed)

ALL_TICKERS = IBOVESPA + US_ETFS

print(f"Universe: {len(IBOVESPA)} B3 stocks + {len(US_ETFS)} US ETFs = {len(ALL_TICKERS)} assets")
print(f"B3:  {', '.join(IBOVESPA)}")
print(f"ETF: {', '.join(US_ETFS)}")


## 3. Fetch Historical Price Data

Fetching 5 years of adjusted daily OHLCV from Yahoo Finance via `mybquant.data.get_prices()`.

In [ ]:

# ── 3. Fetch OHLCV ────────────────────────────────────────────────────────
START = "2020-01-01"
END   = "today"

print(f"Fetching {len(ALL_TICKERS)} tickers from {START} → {END}...")
raw_prices = data.get_prices(ALL_TICKERS, start=START, end=END)
print(f"Shape: {raw_prices.shape}")
print(f"Index range: {raw_prices.index[0].date()} → {raw_prices.index[-1].date()}")
raw_prices.tail(3)


In [ ]:

# ── Extract Adjusted Close Prices ─────────────────────────────────────────
# yfinance multi-ticker returns MultiIndex columns: (field, ticker)
if isinstance(raw_prices.columns, pd.MultiIndex):
    adj_close = raw_prices["Adj Close"].copy()
else:
    # Single ticker fallback
    adj_close = raw_prices[["Adj Close"]].rename(columns={"Adj Close": ALL_TICKERS[0]})

# Drop columns with > 30% missing data
valid_cols = adj_close.columns[adj_close.isna().mean() < 0.30]
adj_close = adj_close[valid_cols].ffill()
print(f"Valid tickers after filtering: {list(valid_cols)}")
adj_close.tail(3)


## 4. Returns Analysis

Log returns: $r_t = \ln(P_t / P_{t-1})$

In [ ]:

# ── 4. Returns & Risk Metrics ─────────────────────────────────────────────
log_returns = np.log(adj_close / adj_close.shift(1)).dropna()

def risk_metrics(returns: pd.Series, name: str = "") -> dict:
    """Compute annualized risk metrics for a return series."""
    r = returns.dropna()
    ann_ret = r.mean() * TRADING_DAYS
    ann_vol = r.std() * np.sqrt(TRADING_DAYS)
    sharpe  = (ann_ret - RISK_FREE_RATE) / ann_vol if ann_vol > 0 else 0
    cum     = (1 + r).cumprod()
    max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()
    return {
        "Asset":      name,
        "Ann. Return":f"{ann_ret:.1%}",
        "Ann. Vol":   f"{ann_vol:.1%}",
        "Sharpe":     f"{sharpe:.2f}",
        "Max DD":     f"{max_dd:.1%}",
        "N days":     len(r),
    }

stats = pd.DataFrame([
    risk_metrics(log_returns[col], col.replace(".SA", ""))
    for col in log_returns.columns
]).set_index("Asset")

print("📊 Annualized Performance Metrics (Log Returns)\n")
print(stats.to_string())


In [ ]:

# ── Normalized Price Chart ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
normalized = (adj_close / adj_close.iloc[0]) * 100
for col in normalized.columns:
    label = col.replace(".SA", "")
    lw = 2.5 if col in ["PETR4.SA", "VALE3.SA", "SPY"] else 1
    alpha = 0.9 if col in ["PETR4.SA", "VALE3.SA", "SPY"] else 0.4
    ax.plot(normalized.index, normalized[col], label=label, linewidth=lw, alpha=alpha)

ax.set_title("Normalized Prices (Base = 100)", fontsize=13, pad=12)
ax.set_ylabel("Normalized Price")
ax.legend(fontsize=7, ncol=4, loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()


In [ ]:

# ── Correlation Heatmap ───────────────────────────────────────────────────
corr = log_returns.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    vmax=1.0,
    vmin=-1.0,
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_title("Returns Correlation Matrix", fontsize=13, pad=12)
ax.set_xticklabels([t.replace(".SA", "") for t in corr.columns], rotation=45, ha="right", fontsize=8)
ax.set_yticklabels([t.replace(".SA", "") for t in corr.index], rotation=0, fontsize=8)
plt.tight_layout()
plt.show()



## 3. Factor Analysis

Compute cross-sectional factor scores for the Brazilian universe using the `mybquant` quant engine.

**Factors computed:**
| Factor | Description |
|---|---|
| `momentum_12_1` | 12-month return skipping last month (formation 252→21 days) |
| `momentum_1` | 1-month short-term reversal |
| `low_vol_12` | Negative 12-month realized volatility |
| `size` | Negative log-price proxy for size |


In [ ]:

from mybquant import quant

ALL_FACTORS = ["momentum_12_1", "momentum_1", "low_vol_12", "size"]

# Compute factor scores using the FactorEngine (adj_close already fetched above)
factor_scores = quant.factors.compute(
    universe=IBOVESPA,
    factors=ALL_FACTORS,
    start="2020-01-01",
    end=None,  # up to today
    as_of=None,  # use latest available date
)

print(f"Factor scores computed for {len(factor_scores)} assets on date: {list(factor_scores.values())[0].index[-1].date() if factor_scores else 'N/A'}")
print()

# Display as a clean DataFrame
scores_df = pd.DataFrame(
    {ticker: {f: factor_scores[ticker][f] for f in ALL_FACTORS} for ticker in factor_scores if ticker in IBOVESPA}
).T.rename(columns=str)

scores_df.index = [t.replace(".SA", "") for t in scores_df.index]
print("Cross-Sectional Factor Scores (Z-scored, winsorized 1%/99%):")
print(scores_df.round(3).to_string())


In [ ]:

# ── Factor Z-Score Bar Charts ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

factor_labels = {
    "momentum_12_1": "12-1 Month Momentum",
    "momentum_1":    "1-Month Reversal",
    "low_vol_12":    "Low Volatility (12M)",
    "size":          "Size (Neg Log-Price)",
}

for ax, factor in zip(axes, ALL_FACTORS):
    if factor not in scores_df.columns:
        ax.set_visible(False)
        continue
    vals = scores_df[factor].sort_values(ascending=True)
    colors = ["#10b981" if v >= 0 else "#ef4444" for v in vals]
    ax.barh(vals.index, vals.values, color=colors, edgecolor="none", height=0.6)
    ax.axvline(0, color="white", linewidth=0.8, alpha=0.5)
    ax.set_title(factor_labels.get(factor, factor), fontsize=10, fontweight="bold")
    ax.set_xlabel("Z-Score", fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="x", alpha=0.1)

fig.suptitle("Factor Scores — IBOVESPA Universe", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()



## 4. Backtest — Long/Short Momentum Strategy

Run a cross-sectional momentum backtest on the IBOVESPA universe:
- **Long** the top-3 momentum stocks each month
- **Short** the bottom-3 momentum stocks each month
- Rebalance monthly with 10 bps commission per side


In [ ]:

from mybquant.quant import backtest as bt
from backend.quant.backtester import Backtester
from backend.quant.factors import FactorEngine
from strategies.momentum import MomentumStrategy

# Build factor engine directly for the backtest
factor_engine = FactorEngine()
strategy = MomentumStrategy(params={"n_long": 3, "n_short": 3})

backtester = Backtester(commission_bps=10, initial_capital=100_000)
result = backtester.run(
    strategy=strategy,
    prices=adj_close[IBOVESPA],
    start="2021-01-01",
    end=None,
)

print(result.summary())


In [ ]:

# ── Equity Curve ──────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), gridspec_kw={"height_ratios": [3, 1]})

equity = result.equity_curve
drawdown_series = (equity / equity.cummax() - 1) * 100

ax1.plot(equity.index, equity.values, color="#3b82f6", linewidth=1.5, label="Strategy NAV")
ax1.fill_between(equity.index, equity.values, equity.iloc[0], alpha=0.08, color="#3b82f6")
ax1.axhline(equity.iloc[0], color="white", linewidth=0.6, linestyle="--", alpha=0.4)
ax1.set_ylabel("Portfolio Value (R$)")
ax1.set_title("Long/Short Momentum Strategy — Equity Curve", fontsize=12)
ax1.legend()
ax1.grid(alpha=0.12)

ax2.fill_between(drawdown_series.index, drawdown_series.values, 0, color="#ef4444", alpha=0.6)
ax2.set_ylabel("Drawdown (%)")
ax2.set_xlabel("")
ax2.grid(alpha=0.12)

plt.tight_layout()
plt.show()



## 5. Portfolio Optimization

Construct a **long-only** mean-variance optimal portfolio from the IBOVESPA universe using PyPortfolioOpt.

- **Risk-free rate**: 11.75% a.a. (approximate Selic)
- **Max weight per asset**: 40%
- Methods tested: `max_sharpe`, `min_volatility`, `risk_parity`


In [ ]:

from mybquant import portfolio

# Max-Sharpe portfolio
opt_sharpe = portfolio.optimize(
    tickers=IBOVESPA,
    method="max_sharpe",
    start="2020-01-01",
    constraints={"max_weight": 0.40, "min_weight": 0.0},
)
print("=== Max Sharpe Portfolio ===")
print(f"Expected Return:  {opt_sharpe.expected_return:.2%}")
print(f"Volatility:       {opt_sharpe.volatility:.2%}")
print(f"Sharpe Ratio:     {opt_sharpe.sharpe_ratio:.3f}\n")
weights_sharpe = pd.Series(opt_sharpe.weights).rename(lambda t: t.replace(".SA", ""))
print(weights_sharpe[weights_sharpe > 0.01].sort_values(ascending=False).apply("{:.1%}".format))


In [ ]:

# ── Portfolio Weights Comparison ──────────────────────────────────────────
opt_minvol = portfolio.optimize(
    tickers=IBOVESPA, method="min_volatility", start="2020-01-01",
    constraints={"max_weight": 0.40},
)
opt_rp = portfolio.optimize(
    tickers=IBOVESPA, method="risk_parity", start="2020-01-01",
)

methods = {
    "Max Sharpe":      pd.Series(opt_sharpe.weights),
    "Min Volatility":  pd.Series(opt_minvol.weights),
    "Risk Parity":     pd.Series(opt_rp.weights),
}
weights_df = pd.DataFrame(methods)
weights_df.index = [t.replace(".SA", "") for t in weights_df.index]
weights_df = weights_df[(weights_df > 0.005).any(axis=1)].sort_values("Max Sharpe", ascending=False)

ax = weights_df.plot(
    kind="bar", figsize=(13, 5),
    color=["#3b82f6", "#10b981", "#f59e0b"],
    edgecolor="none", width=0.75,
)
ax.set_title("Portfolio Weights by Method", fontsize=12)
ax.set_ylabel("Weight")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(loc="upper right")
ax.tick_params(axis="x", rotation=30, labelsize=9)
ax.grid(axis="y", alpha=0.15)
plt.tight_layout()
plt.show()

print("\nMetrics Comparison:")
cmp = pd.DataFrame({
    "Method":         ["Max Sharpe",         "Min Volatility",         "Risk Parity"],
    "Exp. Return":    [opt_sharpe.expected_return, opt_minvol.expected_return, opt_rp.expected_return],
    "Volatility":     [opt_sharpe.volatility,      opt_minvol.volatility,      opt_rp.volatility],
    "Sharpe":         [opt_sharpe.sharpe_ratio,    opt_minvol.sharpe_ratio,    opt_rp.sharpe_ratio],
}).set_index("Method")
print(cmp.applymap(lambda x: f"{x:.3f}" if isinstance(x, float) else x))



## Summary

This notebook covered the full **Phase 1** research workflow of the BQUANT platform:

| Step | Module | Result |
|------|--------|--------|
| 1. Data Fetch | `mybquant.data` | OHLCV for 15 tickers via yfinance + Redis cache |
| 2. Returns Analysis | `pandas / numpy` | Annualized return, vol, Sharpe, max drawdown |
| 3. Factor Scores | `mybquant.quant.factors` | 4 cross-sectional factors, z-scored + winsorized |
| 4. Backtest | `mybquant.quant.backtest` | L/S momentum strategy, monthly rebalance, 10 bps cost |
| 5. Optimization | `mybquant.portfolio` | Max Sharpe / Min Vol / Risk Parity compared |

**Next steps:**
- `notebooks/02_factor_ic_analysis.ipynb` — Information Coefficient (IC) and factor decay analysis
- `notebooks/03_backtest_tearsheet.ipynb` — Full pyfolio tearsheet for the momentum strategy
- `notebooks/04_portfolio_construction.ipynb` — Black-Litterman and robust optimization
- Start the Docker stack: `make up` then visit `http://localhost` for the dashboard
